# Sesión 1 — Ingesta y Arquitectura de Tablas
**FLACSO Ecuador — Estructura Social y Espacial de los Datos y su Arquitectura**  
Docente: Fausto Jácome Pérez

---

## Antes de empezar

Instala las dependencias si aún no lo has hecho:

```bash
pip install -r requirements.txt
```

Descarga los datos de la ENDI desde los links en `Materiales/Insumos/ENDI/links_descarga.txt`  
y colócalos en `Materiales/Insumos/ENDI/BDD_ENDI_R2_rds/`.

## 1. Carga de datos

La ENDI se distribuye en formato `.rds` (nativo de R).  
Para leerlo desde Python usamos `pyreadr`, que devuelve un DataFrame de pandas.  
Luego lo convertimos a Polars, que es la librería principal del curso.

In [ ]:
import pyreadr
import polars as pl

ruta = "../Materiales/Insumos/ENDI/BDD_ENDI_R2_rds/BDD_ENDI_R2_f1_personas.rds"

df = pl.from_pandas(
    pyreadr.read_r(ruta)[None]
)

print("Carga exitosa")

## 2. Exploración inicial

### 2.1 Forma de la tabla

In [ ]:
# Número de filas y columnas
print(f"Filas:    {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

### 2.2 Esquema: nombres de columnas y tipos

In [ ]:
# Schema completo
print(df.schema)

In [ ]:
# Como tabla legible
pl.DataFrame({
    "columna": df.columns,
    "tipo": [str(t) for t in df.dtypes]
})

### 2.3 Primeras filas

In [ ]:
df.head(5)

## 3. Variables numéricas

`describe()` devuelve: `count`, `null_count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`

In [ ]:
# Resumen de todas las variables
df.describe()

In [ ]:
# Solo columnas numéricas
df.select(pl.col(pl.NUMERIC_DTYPES)).describe()

## 4. Variables categóricas

In [ ]:
# Categorías únicas y conteo para variables clave
cat_cols = ["area", "region"]

for col in cat_cols:
    print(f"\n── {col} ──")
    print(df[col].value_counts(sort=True))

## 5. Datos perdidos

En la ENDI los missings no son siempre aleatorios — pueden indicar que  
un registro no aplica a cierta sección del formulario.

In [ ]:
# Conteo de nulos por columna
nulos = df.null_count().transpose(
    include_header=True,
    column_names=["n_nulos"]
).with_columns(
    (pl.col("n_nulos") / len(df) * 100).alias("pct_nulo")
).sort("pct_nulo", descending=True)

# Mostrar solo columnas con al menos un nulo
nulos.filter(pl.col("n_nulos") > 0)

## 6. Preguntas para reflexionar

1. ¿Qué tipo de dato tiene `fexp`? ¿Tiene sentido ese tipo para un factor de expansión?
2. ¿Qué tipo de dato tiene `prov` (código de provincia)? ¿Debería ser numérico o categórico?
3. ¿Qué columnas tienen más missings? ¿A qué sección del formulario corresponden?
4. ¿Cuántas personas hay por área (urbana/rural)? ¿Y por región?